In [76]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [77]:
def preprocessing_Timestamp(df):
    # Convert TIMESTAMP to datetime
    df["TIMESTAMP"] = pd.to_datetime(df["TIMESTAMP"])
    
    # Extract date components
    df["Month"] = df["TIMESTAMP"].dt.month
    df["Day"] = df["TIMESTAMP"].dt.day
    df["Hour"] = df["TIMESTAMP"].dt.hour
    df["Minute"] = df["TIMESTAMP"].dt.minute

    df = df.drop(["TIMESTAMP"], axis = 1) # TIMESTAMP 열 삭제
    
    return df

train = preprocessing_Timestamp(train)
test = preprocessing_Timestamp(test)

In [78]:
from sklearn.preprocessing import LabelEncoder

def preprocessing_Line_Product(df):
    # 'LINE'과 'PRODUCT_CODE_encoded'의 조합을 하나의 문자열로 결합
    df['LINE_PRODUCT_COMBINATION'] = df['LINE'].astype(str) + '_' + df['PRODUCT_CODE'].astype(str)

    # LabelEncoder를 사용하여 고유한 숫자 레이블 부여
    label_encoder = LabelEncoder()
    df['LINE_PRODUCT_LABEL'] = label_encoder.fit_transform(df['LINE_PRODUCT_COMBINATION'])

    df = df.drop(["LINE", "PRODUCT_CODE", "LINE_PRODUCT_COMBINATION"], axis = 1)

    return df

train = preprocessing_Line_Product(train)
test = preprocessing_Line_Product(test)

In [79]:
train_x = train.drop(columns=['PRODUCT_ID', 'Y_Quality'])
train_y = train['Y_Class']

test_x = test.drop(columns=['PRODUCT_ID'])

In [80]:
from sklearn.preprocessing import MinMaxScaler

# X 컬럼 추출
x_cols = train_x.columns[train_x.columns.str.startswith('X')].tolist()

# 그룹 매핑 함수
def map_line_group(label):
    if label in [0, 1]:
        return 0
    elif label in [2, 3]:
        return 1
    elif label in [4, 6]:
        return 2
    elif label in [5, 7]:
        return 3
    else:
        return -1

# 학습 데이터에 그룹 라벨 추가
train_x['LINE_GROUP_LABEL'] = train_x['LINE_PRODUCT_LABEL'].apply(map_line_group)

# 그룹별 스케일러 저장 딕셔너리
group_scalers = {}

# 그룹별 정규화 및 스케일러 저장
for group_label in train_x['LINE_GROUP_LABEL'].unique():
    idx = train_x['LINE_GROUP_LABEL'] == group_label
    scaler = MinMaxScaler()
    train_x.loc[idx, x_cols] = scaler.fit_transform(train_x.loc[idx, x_cols])
    group_scalers[group_label] = scaler  # 저장

# 보조 컬럼 제거
train_x.drop(columns=['LINE_GROUP_LABEL'], inplace=True)

# 테스트 데이터에도 그룹 라벨 생성
test_x['LINE_GROUP_LABEL'] = test_x['LINE_PRODUCT_LABEL'].apply(map_line_group)

# 그룹별 정규화 적용
for group_label in test_x['LINE_GROUP_LABEL'].unique():
    idx = test_x['LINE_GROUP_LABEL'] == group_label
    scaler = group_scalers[group_label]  # 학습된 스케일러 가져오기
    test_x.loc[idx, x_cols] = scaler.transform(test_x.loc[idx, x_cols])

# 필요 시 제거
test_x.drop(columns=['LINE_GROUP_LABEL'], inplace=True)

c:\Users\twoh0\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_array_api.py:776: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
c:\Users\twoh0\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_array_api.py:793: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmax(X, axis=axis))
c:\Users\twoh0\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_array_api.py:776: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
c:\Users\twoh0\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_array_api.py:793: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmax(X, axis=axis))
c:\Users\twoh0\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_array_api.py:776: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
c:\Users\twoh0\AppDa

In [81]:
#빈 slot을 모두 평균으로 채움
train_x = train_x.fillna(train_x.mean())
test_x = test_x.fillna(train_x.mean())

train_x = train_x.dropna(axis = 1)
test_x = test_x.dropna(axis = 1)

In [82]:
# X로 시작하는 수치형 컬럼 선택
x_cols = train_x.columns[train_x.columns.str.startswith('X')].tolist()

# 상관관계 계산
corr = train_x[x_cols + ['Y_Class']].corr()

# Y_Class 기준 상관계수 정렬 (상위 몇 개만 보기 원할 수도 있음)
corr_with_target = corr['Y_Class'].drop('Y_Class').sort_values(key = abs, ascending=False)

top_features = [i for i in corr_with_target.index if abs(corr_with_target[i]) > 0.05]
top_features_sorted = [col for col in train_x.columns if col in top_features]

train_x = train_x[top_features_sorted + ['Month', 'Day', 'Hour', 'Minute', 'LINE_PRODUCT_LABEL']]
test_x = test_x[top_features_sorted + ['Month', 'Day', 'Hour', 'Minute', 'LINE_PRODUCT_LABEL']] 

test_x
# test_x

,X_2,X_5,X_8,X_24,X_38,X_44,X_56,X_62,X_73,X_90,...,X_2865,X_2866,X_2867,X_2869,X_2870,Month,Day,Hour,Minute,LINE_PRODUCT_LABEL
0,0.466667,0.00000,0.000000,0.00000,0.000000,0.530612,0.644444,0.570093,0.679012,0.500000,...,0.589759,0.664555,0.592741,0.719083,0.275426,9,9,2,1,7
1,0.400000,1.00000,0.000000,0.00000,1.000000,0.142857,0.000000,0.074766,0.802469,0.000000,...,0.589759,0.664555,0.592741,0.719083,0.275426,9,9,2,9,5
2,0.533333,1.00000,0.000000,0.00000,1.000000,0.142857,0.000000,0.074766,0.802469,0.000000,...,0.589759,0.664555,0.592741,0.719083,0.275426,9,9,8,42,5
3,0.542605,0.39255,0.048711,0.13467,0.590974,0.406615,0.505571,0.497577,0.507610,0.526743,...,0.725000,0.664555,0.592741,0.719083,0.275426,9,9,10,56,0
4,0.542605,0.39255,0.048711,0.13467,0.590974,0.406615,0.505571,0.497577,0.507610,0.526743,...,0.725000,0.664555,0.592741,0.719083,0.275426,9,9,11,4,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
305,0.266667,0.00000,0.000000,1.00000,0.250000,0.714286,0.711111,0.775701,0.345679,0.500000,...,0.589759,0.664555,0.592741,0.719083,0.275426,11,5,11,18,7
306,0.600000,1.00000,0.000000,0.00000,1.000000,0.326531,0.200000,0.233645,0.530864,0.500000,...,0.589759,0.664555,0.592741,0.719083,0.275426,11,5,16,39,5
307,0.266667,0.00000,0.000000,1.00000,0.000000,0.714286,0.711111,0.775701,0.345679,0.500000,...,0.589759,0.664555,0.592741,0.719083,0.275426,11,5,16,47,7
308,0.533333,0.00000,0.000000,1.00000,0.250000,0.714286,0.711111,0.775701,0.382716,0.500000,...,0.589759,0.664555,0.592741,0.719083,0.275426,11,5,20,53,7


In [ ]:
from lightgbm import LGBMClassifier
from sklearn.ensemble import VotingClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score# 정확도 함수
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(train_x, train_y, test_size=0.2, random_state = 100)

# params = {
#     'num_leaves': [70, 80, 90, 100, 110, 120, 130],
#     'learning_rate': [0.05, 0.1],
#     'n_estimators': [70, 80, 90, 100, 110, 120, 130],
# }
# grid = GridSearchCV(lgbm.LGBMClassifier(random_state=37), params, scoring='f1_macro', cv=3)
# grid.fit(train_x, train_y)
# print(grid.best_params_)

model = LGBMClassifier()
et_cls = ExtraTreesClassifier(n_estimators = 500, min_samples_leaf = 5, min_samples_split = 7, max_features = 1431)
rf_cls = RandomForestClassifier(n_estimators = 500, min_samples_leaf = 5, min_samples_split = 7, max_features = 1431)
lg_cls = model.fit(train_x, train_y)

# 모델 voting

voting = VotingClassifier(
    estimators=[
        ('et', et_cls),
        ('rf', rf_cls),
        ('lg', lg_cls)
    ]
)

voting.fit(train_x, train_y)
Y_Label = voting.predict(test_x)

submit = pd.read_csv("sample_submission.csv")
submit['Y_Class'] = Y_Label
submit.to_csv("submission.csv", index=False)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013893 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 29153
[LightGBM] [Info] Number of data points in the train set: 598, number of used features: 1370
[LightGBM] [Info] Start training from score -1.916254
[LightGBM] [Info] Start training from score -0.384778
[LightGBM] [Info] Start training from score -1.758862
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No